# 06 · Rigid-Body Dynamics in 3D

### Recap & why now
Five notebooks of orientation machinery, and none of it knows what a newton is. This
notebook writes the physics: two equations, one for how the centre of mass moves and
one for how the body rotates, joined by the rotation matrix from Notebook 04.

By the end you have a working nonlinear 6-DOF simulator. It just has to be flown by
commanding force and torque directly, which Notebook 07 fixes.

### Learning objectives
1. Write the **13-element state** and say which frame each block lives in.
2. Derive translational acceleration from thrust and gravity, with the ENU signs right.
3. Apply **Euler's rigid-body equation** and explain the gyroscopic term.
4. Assemble one `quad_dynamics` function and **verify hover** numerically.
5. Compare Euler and RK4 integration and choose a time step on evidence.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection that every figure here needs.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print matrices with 3 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=3, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Orientation toolkit, built up over Notebooks 02-05 ==================

def quat_normalize(q):
    """Force |q| = 1. Integration drifts off the unit sphere; this pulls it back."""
    q = np.asarray(q, float)
    return q/np.linalg.norm(q)

def quat_multiply(a, b):
    """Hamilton product a (x) b: 'do b first, then a', the same reading order as matrices."""
    aw, ax, ay, az = a
    bw, bx, by, bz = b
    return np.array([aw*bw - ax*bx - ay*by - az*bz,     # Scalar part.
                     aw*bx + ax*bw + ay*bz - az*by,     # Vector part, x.
                     aw*by - ax*bz + ay*bw + az*bx,     #              y.
                     aw*bz + ax*by - ay*bx + az*bw])    #              z.

def quat_conjugate(q):
    """Flip the vector part — for a unit quaternion this is the INVERSE rotation."""
    return np.array([q[0], -q[1], -q[2], -q[3]])

def quat_to_rotmat(q):
    """The body-to-world rotation matrix that this quaternion represents."""
    w, x, y, z = quat_normalize(q)
    return np.array([[1-2*(y*y+z*z),   2*(x*y-w*z),   2*(x*z+w*y)],
                     [  2*(x*y+w*z), 1-2*(x*x+z*z),   2*(y*z-w*x)],
                     [  2*(x*z-w*y),   2*(y*z+w*x), 1-2*(x*x+y*y)]])

def euler_to_quat(roll, pitch, yaw):
    """ZYX Euler angles -> quaternion. Used to SET a pose, never to store one."""
    cr, sr = np.cos(roll/2), np.sin(roll/2)
    cp, sp = np.cos(pitch/2), np.sin(pitch/2)
    cy, sy = np.cos(yaw/2), np.sin(yaw/2)
    return np.array([cr*cp*cy + sr*sp*sy, sr*cp*cy - cr*sp*sy,
                     cr*sp*cy + sr*cp*sy, cr*cp*sy - sr*sp*cy])

def quat_to_euler(q):
    """Quaternion -> roll, pitch, yaw. For DISPLAY only — never as simulator state."""
    w, x, y, z = quat_normalize(q)
    return np.array([np.arctan2(2*(w*x + y*z), 1 - 2*(x*x + y*y)),
                     np.arcsin(np.clip(2*(w*y - z*x), -1, 1)),      # clip guards against 1+1e-16.
                     np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))])

def quat_from_rotmat(R):
    """Rotation matrix -> quaternion. Four branches, so we never divide by a small number."""
    tr = np.trace(R)
    if tr > 0:
        s_ = np.sqrt(tr + 1.0)*2
        q = np.array([0.25*s_, (R[2,1]-R[1,2])/s_, (R[0,2]-R[2,0])/s_, (R[1,0]-R[0,1])/s_])
    elif R[0,0] > R[1,1] and R[0,0] > R[2,2]:
        s_ = np.sqrt(1.0 + R[0,0] - R[1,1] - R[2,2])*2
        q = np.array([(R[2,1]-R[1,2])/s_, 0.25*s_, (R[0,1]+R[1,0])/s_, (R[0,2]+R[2,0])/s_])
    elif R[1,1] > R[2,2]:
        s_ = np.sqrt(1.0 + R[1,1] - R[0,0] - R[2,2])*2
        q = np.array([(R[0,2]-R[2,0])/s_, (R[0,1]+R[1,0])/s_, 0.25*s_, (R[1,2]+R[2,1])/s_])
    else:
        s_ = np.sqrt(1.0 + R[2,2] - R[0,0] - R[1,1])*2
        q = np.array([(R[1,0]-R[0,1])/s_, (R[0,2]+R[2,0])/s_, (R[1,2]+R[2,1])/s_, 0.25*s_])
    return quat_normalize(q)

def axis_angle_to_quat(axis, angle):
    """Build a quaternion from 'rotate by `angle` about `axis`' — the geometric reading."""
    axis = np.asarray(axis, float); axis = axis/np.linalg.norm(axis)
    return np.array([np.cos(angle/2), *(axis*np.sin(angle/2))])

def quat_rotate(q, v):
    """Rotate v from the body frame into the world frame, using the sandwich product."""
    return quat_multiply(quat_multiply(q, np.array([0.0, *v])), quat_conjugate(q))[1:]

# === The vehicle, and how to draw it =====================================

PARAMS = dict(m=1.0, L=0.25,                       # Mass [kg] and hub-to-rotor distance [m].
              I=np.diag([0.01, 0.01, 0.02]),       # Inertia [kg m^2]; yaw is the heavy axis.
              d=0.016,                             # Drag torque per newton of thrust [m].
              T_min=0.0, T_max=6.0)                # What one motor can produce [N].
g = 9.81                                           # Gravity [m/s^2], along world -z.

ARM = PARAMS["L"]/np.sqrt(2)                       # Each rotor sits ARM along body x AND body y.
MOTOR_POS = np.array([[ ARM, -ARM, 0.0],           # M1 front-right.
                      [ ARM,  ARM, 0.0],           # M2 front-left.
                      [-ARM,  ARM, 0.0],           # M3 rear-left.
                      [-ARM, -ARM, 0.0]])          # M4 rear-right.
SPIN = np.array([-1.0, 1.0, -1.0, 1.0])            # +1 = counter-clockwise seen from above.

def draw_quad(ax, position, q, scale=3.0, thrusts=None):
    """Draw the drone: four arms, four rotors, a nose marker and the thrust arrow."""
    R = quat_to_rotmat(q)                          # Body-to-world, so body points become world points.
    for i, mp in enumerate(MOTOR_POS):
        tip = np.asarray(position, float) + R @ (mp*scale)
        seg = np.array([position, tip])
        ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], color="0.35", lw=2)
        shade = "C3" if i in (0, 1) else "C0"      # Front rotors red, rear blue, so the nose is visible.
        if thrusts is not None:
            load = np.clip(thrusts[i]/PARAMS["T_max"], 0, 1)
            shade = plt.cm.YlOrRd(0.3 + 0.7*load)  # Colour by how hard the motor is working.
        ax.plot([tip[0]], [tip[1]], [tip[2]], "o", ms=6, color=shade)
    ax.quiver(*position, *(R[:, 2]*0.9), color="C1", lw=2.2, arrow_length_ratio=0.25)

def set_3d(ax, xlim, ylim, zlim):
    """Equal-ish 3-D axes with explicit limits, so animations do not jitter."""
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
    ax.set_box_aspect([xlim[1]-xlim[0], ylim[1]-ylim[0], zlim[1]-zlim[0]])
    ax.set_xlabel("x — East [m]"); ax.set_ylabel("y — North [m]"); ax.set_zlabel("z — Up [m]")

print("vehicle ready: %.1f kg, hover %.2f N total, %.3f N per motor, thrust/weight %.2f" %
      (PARAMS["m"], PARAMS["m"]*g, PARAMS["m"]*g/4, 4*PARAMS["T_max"]/(PARAMS["m"]*g)))

## 1 · Thirteen numbers

$$x = [\,\underbrace{p}_{3},\; \underbrace{v}_{3},\; \underbrace{q}_{4},\; \underbrace{\omega}_{3}\,]
\;\in\; \mathbb{R}^{13}$$

Six degrees of freedom, thirteen numbers — the extra one is the quaternion's redundancy,
which Notebook 03 showed is the price of having no singularity.

Which frame each block lives in matters enormously: $p$ and $v$ are in the **world**,
$q$ maps **body → world**, and $\omega$ is in the **body**, because that is what a gyro
measures and what Euler's equation is written in.

In [ ]:
POS, VEL, QUAT, OMEGA = slice(0, 3), slice(3, 6), slice(6, 10), slice(10, 13)

def make_state(p=(0, 0, 0), v=(0, 0, 0), q=(1, 0, 0, 0), w=(0, 0, 0)):
    """Assemble the 13-element state vector; defaults are level, still, at the origin."""
    return np.concatenate([p, v, q, w]).astype(float)

s = make_state(p=(1.0, 2.0, 3.0), q=euler_to_quat(np.deg2rad(10), 0, np.deg2rad(45)), w=(0, 0, 0.5))
print("position [m]      ", np.round(s[POS], 3), "  (world frame)")
print("velocity [m/s]    ", np.round(s[VEL], 3), "  (world frame)")
print("quaternion        ", np.round(s[QUAT], 4), " -> rpy", np.round(np.degrees(quat_to_euler(s[QUAT])), 1), "deg")
print("body rates [rad/s]", np.round(s[OMEGA], 3), "  (body frame)")
print("\n|q| = %.12f — must stay 1, or the rotation matrix stops being a rotation." % np.linalg.norm(s[QUAT]))

## 2 · Translation

$$m\dot v = \underbrace{R(q)\begin{bmatrix}0\\0\\T\end{bmatrix}}_{\text{thrust, rotated}}
+ \underbrace{\begin{bmatrix}0\\0\\-mg\end{bmatrix}}_{\text{gravity, ENU}}$$

Three things to keep straight. Thrust is **always** $[0,0,T]$ in the body frame —
rotors have no idea which way is up. $T \ge 0$, because a propeller cannot pull.
And gravity is **negative** in ENU, which is the single sign that most often sends a
simulated drone to the moon.

In [ ]:
def translational_accel(q, T, p=PARAMS, f_ext=np.zeros(3)):
    """World-frame acceleration from thrust magnitude, gravity, and any external push."""
    thrust_world = quat_to_rotmat(q) @ np.array([0.0, 0.0, T])   # Body -> world.
    return (thrust_world + np.array([0.0, 0.0, -p["m"]*g]) + f_ext)/p["m"]

T_hover = PARAMS["m"]*g
q_level = np.array([1.0, 0.0, 0.0, 0.0])
print("  situation              thrust [N]   acceleration [m/s^2]")
for label, T, q_ in [("hover, level      ", T_hover, q_level),
                     ("20% more thrust   ", 1.2*T_hover, q_level),
                     ("motors off        ", 0.0, q_level),
                     ("hover thrust, 20° ", T_hover, euler_to_quat(np.deg2rad(20), 0, 0))]:
    print("  %s %10.3f %20s" % (label, T, np.round(translational_accel(q_, T), 3)))

print("\nRow 1 is exactly zero, not merely small — mg minus mg cancels in floating point too.")
print("Row 4 slides sideways AND sinks, because a tilted drone spends part of its lift.")

## 3 · Rotation

$$I\dot\omega + \omega \times (I\omega) = \tau
\qquad\Longrightarrow\qquad
\dot\omega = I^{-1}\left[\tau - \omega \times (I\omega)\right]$$

$I$ is rotation's version of mass, and unlike mass it is directional. The cross-product
term is **gyroscopic coupling**: a body already spinning about two axes carries angular
momentum that is not aligned with $\omega$, and moving that around produces an apparent
torque on the third axis.

Our $I_{zz}$ is twice $I_{xx}$, because yawing swings all four motors out at full arm
length while rolling only swings two. That is why yaw is the sluggish axis on every
multirotor.

In [ ]:
I = PARAMS["I"]
print("inertia [kg m^2]:", np.diag(I))
print("\n  axis    same 0.05 N m torque gives")
for i, name in enumerate(["roll ", "pitch", "yaw  "]):
    tau = np.zeros(3); tau[i] = 0.05
    print("  %s %14.2f rad/s^2" % (name, np.linalg.solve(I, tau)[i]))

print("\n  body rates [rad/s]      gyroscopic torque [N m]")
for w in [np.array([2.0, 0, 0]), np.array([2.0, 1.0, 0]), np.array([2.0, 0, 2.0]), np.array([4.0, 3.0, 2.0])]:
    print("  %-22s %s" % (np.round(w, 1), np.round(np.cross(w, I @ w), 4)))
print("\nTwo rows are exactly zero, for different reasons. A single-axis spin makes omega and")
print("I*omega parallel. But [2,1,0] is zero TOO, because I_xx = I_yy on this symmetric frame —")
print("any roll-plus-pitch combination stays parallel. You need yaw rate before coupling appears.")

## 4 · The whole model, and an integrator

Both equations in one function, taking a **wrench** — total thrust and three torques —
as its input. That is not something you can send to real hardware; Notebook 07 replaces
it with four motor thrusts.

For the integrator, RK4. The next cell measures why: at the same step size it is
millions of times more accurate than Euler, for four times the arithmetic.

In [ ]:
def dynamics_wrench(state, u, p=PARAMS):
    """x_dot for the 13-state quadcopter. u = [T, tau_x, tau_y, tau_z]."""
    q = quat_normalize(state[QUAT]); w = state[OMEGA]
    T, tau = u[0], np.asarray(u[1:], float)
    v_dot = translational_accel(q, T, p)                        # Section 2.
    q_dot = 0.5*quat_multiply(q, np.array([0.0, *w]))           # Notebook 05.
    w_dot = np.linalg.solve(p["I"], tau - np.cross(w, p["I"] @ w))   # Section 3.
    return np.concatenate([state[VEL], v_dot, q_dot, w_dot])

def step_euler(state, u, dt, p=PARAMS):
    s_ = state + dt*dynamics_wrench(state, u, p)
    s_[QUAT] = quat_normalize(s_[QUAT]); return s_

def step_rk4(state, u, dt, p=PARAMS):
    k1 = dynamics_wrench(state, u, p); k2 = dynamics_wrench(state + 0.5*dt*k1, u, p)
    k3 = dynamics_wrench(state + 0.5*dt*k2, u, p); k4 = dynamics_wrench(state + dt*k3, u, p)
    s_ = state + dt/6*(k1 + 2*k2 + 2*k3 + k4)
    s_[QUAT] = quat_normalize(s_[QUAT]); return s_

u_test = [1.10*T_hover, 0.02, 0.0, 0.0]            # Climbing while rolling: real 3-D motion.
def march(stepper, dt, T_end=1.5):
    s_ = make_state()
    for _ in range(int(round(T_end/dt))):
        s_ = stepper(s_, u_test, dt)
    return s_
ref = march(step_rk4, 0.0002)                      # A very fine RK4 run stands in for the truth.

print("  step size      Euler position error       RK4 position error")
for dt in (0.02, 0.01, 0.005):
    print("   %.3f s %20.3e m %20.3e m" %
          (dt, np.linalg.norm(march(step_euler, dt)[POS] - ref[POS]),
           np.linalg.norm(march(step_rk4, dt)[POS] - ref[POS])))
print("\nWe use RK4 with dt = 0.005 s for the rest of the project.")

## 🧪 Try it yourself

**E1.** Hover came out exactly zero. Why exactly, rather than approximately — and what
would a non-zero answer have told you?

**E2.** Apply a constant roll torque with hover thrust for four seconds. Predict what
happens to the altitude before running it.

In [ ]:
# --- Solution E1 ---
print("E1: at hover the thrust term is R(identity) @ [0,0,mg] = [0,0,mg] and gravity is [0,0,-mg].")
print("    Adding a number to its own negative is exact in floating point, so the result is 0.0")
print("    and not 1e-16. A non-zero answer would mean one of three things: the mass used for")
print("    thrust differs from the mass used for weight, the quaternion was not exactly identity,")
print("    or g appears with two different values somewhere. All three are worth catching early.")

# --- Solution E2 ---
def simulate_wrench(u_fn, T_end, dt=0.005, s0=None):
    """Fly with a time-varying wrench, logging everything."""
    s_ = make_state() if s0 is None else np.array(s0, float)
    ts, xs = [0.0], [s_.copy()]
    for k in range(int(T_end/dt)):
        s_ = step_rk4(s_, np.asarray(u_fn(k*dt), float), dt)
        ts.append((k+1)*dt); xs.append(s_.copy())
    return np.array(ts), np.array(xs)

t, X = simulate_wrench(lambda t_: [T_hover, 0.02, 0, 0], 4.0)
rpy = np.degrees(np.array([quat_to_euler(q_) for q_ in X[:, QUAT]]))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 3.0))
a1.plot(t, rpy[:, 0], color="C3", lw=2); a1.axhline(90, color="0.6", ls=":", lw=1.2)
a1.set_xlabel("time [s]"); a1.set_ylabel("roll [deg]"); a1.set_title("Roll runs away")
a2.plot(t, X[:, 2], color="C0", lw=2)
a2.set_xlabel("time [s]"); a2.set_ylabel("altitude [m]"); a2.set_title("Altitude follows it down")
plt.tight_layout(); plt.show()

past = t[np.argmax(np.abs(rpy[:, 0]) > 90)]
print("E2: nothing damps rotation, so the roll rate grows without limit and the drone passes")
print("    90° at t = %.2f s. Past vertical the thrust points partly DOWNWARD and stops fighting" % past)
print("    gravity — altitude ends at %.1f m. There is no restoring force anywhere in these" % X[-1, 2])
print("    equations: a quadcopter is an inverted pendulum with rotors, and everything that makes")
print("    one flyable lives in the controller, not the airframe.")

## 🚁 Mini-project: an uncontrolled tumble

Fly the run from E2 and watch it. The drone starts level and perfectly trimmed; a
constant, small torque is all it takes. Hold on to this clip — the four notebooks after
this one exist to prevent it.

In [ ]:
t, X = simulate_wrench(lambda t_: [T_hover, 0.02, 0, 0], 4.0)
step = 12
fig = plt.figure(figsize=(6.6, 5.4))
ax = fig.add_subplot(111, projection="3d")

def frame(j):
    ax.clear()
    k = step*j
    ax.plot(X[:k+1, 0], X[:k+1, 1], X[:k+1, 2], color="0.75", lw=1.4)   # The path so far.
    draw_quad(ax, X[k, POS], X[k, QUAT], scale=2.5)
    set_3d(ax, (-2, 2), (-14, 2), (-16, 2))
    rpy_k = np.degrees(quat_to_euler(X[k, QUAT]))
    ax.set_title("t = %4.2f s   roll %7.1f°   altitude %6.2f m" % (k*0.005, rpy_k[0], X[k, 2]),
                 fontsize=10)
    ax.view_init(elev=18, azim=-62)
    return []

anim = animation.FuncAnimation(fig, frame, frames=len(X)//step, interval=50, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** These two equations, in exactly this form, run inside
> every drone simulator from a student notebook to a certification rig. Production
> models add rotor spin-up lag, body drag, blade flapping and ground effect — all
> refinements to the force term, none of them changes the skeleton: rotate the thrust,
> add gravity, divide torque by inertia.

**Where next.** We have been commanding torque directly, which no motor accepts.
Notebook 07 replaces the wrench with the four numbers a real flight controller sends.